In [ ]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY") 
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

In [ ]:
# Data ingestion --> from the website we need to scrape the data 
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://www.geeksforgeeks.org/dsa/array-data-structure-guide/")
docs = loader.load()
   

In [ ]:
# Load data --> Docs --> Divide data into chunks --> vectors --> vector embeddings--> vector store db
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,
                                               chunk_overlap = 200)
documents = text_splitter.split_documents(docs)
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vectorstoredb = FAISS.from_documents(documents,embeddings)
vectorstoredb

In [ ]:
query = "Provide me important DSA Concepts"
result = vectorstoredb.similarity_search(query)[0].page_content
result




In [ ]:
# Retrieval chain,document chain 
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate 
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains import create_retrieval_chain
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=1.0,  
    max_tokens=None,
    timeout=None,
    max_retries=2
)

prompt = ChatPromptTemplate.from_template(
    """ Answer the following questions based on the provided context: 
    <context>
    {context}
    </context>
    """
)
document_chain = create_stuff_documents_chain(llm,prompt)
print(document_chain)



In [ ]:
document_chain.invoke({
    "input":"ndian development is not so good in terms of infrasture here",
    "context":[Document(page_content = "In recent years world has experienced global crisis and indian development is not so good in terms of infrasture here")]
})

In [ ]:
# However we want the document to come up before retriever is set up. 
# input --> retriever --> vectorstoredb 
# we convert vector store db to retriever 
retriever =vectorstoredb.as_retriever()

retrieval_chain = create_retrieval_chain(retriever,document_chain)


In [ ]:
# Get response from the llm 
retrieval_chain.invoke({"input":"Rotate an array"})